# 03 - Réseaux de neurones légers

Objectif : faire mieux que la baseline NDWI (F1 de validation = 0,625) avec un modèle qui reste embarquable.

Principe : le réseau produit une **carte de probabilité d'eau** (un nombre par pixel) à partir des
13 bandes de Sentinel-2. La décision d'alerte reste la même que pour la baseline : part de pixels
d'eau hors eau permanente, comparée à un seuil `t_alerte` choisi sur l'entraînement.

Deux réseaux comparés :
- **Petit U-Net**, entraîné de zéro (environ 0,5 million de paramètres) ;
- **U-Net MobileNetV2**, dont l'encodeur est pré-entraîné sur ImageNet (environ 4 millions de paramètres).

Exécution : Google Colab avec GPU (*Exécution > Modifier le type d'exécution > GPU T4*),
puis *Exécution > Tout exécuter* (environ 15 minutes).

In [ ]:
import os, subprocess, sys

if not os.path.isdir("one-look"):
    subprocess.run(["git", "clone", "-q", "https://github.com/lemembre05/one-look.git"], check=True)
else:
    subprocess.run(["git", "-C", "one-look", "pull", "-q"], check=True)
sys.path.insert(0, "one-look")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio", "segmentation-models-pytorch"], check=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from src import donnees, modeles, entrainement, alerte
from src.metriques import scores_alerte

noms = donnees.telecharger("data")
APPAREIL = "cuda" if torch.cuda.is_available() else "cpu"
print(len(noms), "images | calcul sur", APPAREIL, torch.cuda.get_device_name(0) if APPAREIL == "cuda" else "")
if APPAREIL == "cpu":
    print("ATTENTION : pas de GPU. Exécution > Modifier le type d'exécution > GPU T4, puis relancer.")

## 1. Chargement des images en mémoire

In [ ]:
images, labels, perms = [], [], []
for nom in noms:
    s2, label, perm = donnees.lire_image(nom, "data")
    images.append(s2.astype(np.int16))
    labels.append(label.astype(np.int8))
    perms.append(perm)

df = pd.DataFrame({"image": noms, "pays": [donnees.pays(n) for n in noms], "partie": [donnees.partie(n) for n in noms]})
df["part_inondee"] = [donnees.part_inondee_vraie(l.astype(np.float32), p) for l, p in zip(labels, perms)]
SEUIL_ALERTE = 0.05                      # même définition de l'alerte que dans le notebook 02
df["alerte_vraie"] = df["part_inondee"] >= SEUIL_ALERTE

idx = {p: df.index[df["partie"] == p].tolist() for p in ["train", "validation", "test"]}
moyenne, ecart = entrainement.statistiques_bandes(images, idx["train"])
print({p: len(i) for p, i in idx.items()})

## 2. Entraînement

- découpes aléatoires de 256 x 256 pixels, retournées et tournées au hasard (augmentation de données) ;
- perte = entropie croisée + Dice, calculée seulement sur les pixels annotés (les nuages sont ignorés) ;
- à chaque époque, IoU de l'eau sur la validation ; on garde la meilleure époque.

La validation sert donc aussi à choisir l'époque : le test reste la seule mesure totalement indépendante.

In [ ]:
EPOQUES = 30 if APPAREIL == "cuda" else 3

reseaux = {
    "Petit U-Net": modeles.Normalise(modeles.PetitUNet(), moyenne, ecart),
    "U-Net MobileNetV2": modeles.Normalise(modeles.unet_mobilenet(pre_entraine=True), moyenne, ecart),
}
historiques = {}
for nom, m in reseaux.items():
    print(f"\n=== {nom} : {modeles.nb_parametres(m):,} paramètres ===".replace(",", " "))
    historiques[nom] = entrainement.entrainer(m, images, labels, idx["train"], idx["validation"],
                                              APPAREIL, epoques=EPOQUES)
    os.makedirs("modeles", exist_ok=True)
    torch.save(m.state_dict(), f"modeles/{nom.replace(' ', '_')}.pt")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for nom, h in historiques.items():
    ax[0].plot(h["epoque"], h["perte"], label=nom)
    ax[1].plot(h["epoque"], h["iou_validation"], label=nom)
ax[0].set_title("Perte d'entraînement"); ax[1].set_title("IoU de l'eau, validation")
for a in ax: a.set_xlabel("époque"); a.legend()
plt.show()

## 3. De la carte d'eau à l'alerte

Pour chaque image : part de pixels avec une probabilité d'eau > 0,5, hors eau permanente.
Le seuil `t_alerte` est choisi sur l'entraînement, exactement comme pour la baseline.

In [ ]:
train = (df["partie"] == "train").to_numpy()
tous = df.index.tolist()

parts = {"Baseline NDWI": np.array([alerte.part_eau_ndwi(images[k].astype(np.float32), perms[k]) for k in tous])}
seuils = {"Baseline NDWI": alerte.BASELINE_T_ALERTE}
probs = {}
for nom, m in reseaux.items():
    probs[nom] = entrainement.predire(m, images, tous, APPAREIL)
    parts[nom] = np.array([alerte.part_eau_reseau(probs[nom][k], images[k], perms[k]) for k in tous])
    seuils[nom], _ = alerte.choisir_seuil_alerte(df["alerte_vraie"][train], parts[nom][train], SEUIL_ALERTE)

lignes = []
for nom in parts:
    pred = parts[nom] >= seuils[nom]
    for p in ["validation", "test"]:
        sel = (df["partie"] == p).to_numpy()
        s = scores_alerte(df["alerte_vraie"][sel], pred[sel])
        lignes.append({"modele": nom, "partie": p, "t_alerte": seuils[nom], **s})
scores = pd.DataFrame(lignes)
print(scores[["modele", "partie", "t_alerte", "precision", "rappel", "f1", "taux_fausse_alerte",
              "fausses_alertes", "manquees"]].to_string(index=False))

## 4. Qualité des cartes d'eau (IoU par pixel)

L'alerte ne dit pas tout : l'IoU mesure si l'eau est détectée **au bon endroit**.

In [ ]:
iou = {}
for p in ["validation", "test"]:
    t_ndwi = {k: (donnees.ndwi(images[k].astype(np.float32)) > alerte.BASELINE_T_NDWI).astype(np.float16) for k in idx[p]}
    iou[("Baseline NDWI", p)] = entrainement.iou_eau(t_ndwi, labels, idx[p])
    for nom in reseaux:
        iou[(nom, p)] = entrainement.iou_eau(probs[nom], labels, idx[p])
print(pd.Series(iou).unstack().round(3))

## 5. Embarquabilité : taille et temps de calcul

Temps d'inférence d'une image complète (512 x 512 pixels, 13 bandes) sur **un seul cœur** de processeur,
plus proche d'un calculateur embarqué qu'un GPU.

In [ ]:
lignes = [{"modele": "Baseline NDWI", "parametres": 2, "taille_Mo": 0.0, "ms_par_image_1_coeur": None}]
for nom, m in reseaux.items():
    lignes.append({"modele": nom, "parametres": modeles.nb_parametres(m),
                   "taille_Mo": round(modeles.taille_mo(m), 2),
                   "ms_par_image_1_coeur": round(entrainement.latence_cpu_ms(m), 0)})
embarque = pd.DataFrame(lignes)

import time
debut = time.perf_counter()
for k in idx["validation"][:20]:
    alerte.part_eau_ndwi(images[k].astype(np.float32), perms[k])
embarque.loc[0, "ms_par_image_1_coeur"] = round((time.perf_counter() - debut) / 20 * 1000, 1)

tableau = scores.pivot(index="modele", columns="partie", values=["f1", "taux_fausse_alerte"])
tableau.columns = [f"{a}_{b}" for a, b in tableau.columns]
tableau = embarque.set_index("modele").join(tableau)
print(tableau.to_string())
os.makedirs("resultats", exist_ok=True)
tableau.to_csv("resultats/comparaison_03.csv")

## 6. Cas d'échec de la baseline : le réseau fait-il mieux ?

Les trois inondations que la baseline ratait en validation et en test (notebook 02), puis les erreurs
restantes du meilleur réseau.

In [ ]:
meilleur = scores[(scores["partie"] == "validation") & (scores["modele"] != "Baseline NDWI")] \
    .sort_values("f1", ascending=False).iloc[0]["modele"]
print("Meilleur réseau en validation :", meilleur)

def montrer(k, titre):
    s2 = images[k].astype(np.float32)
    vrai = (labels[k] == 1) & ~perms[k]
    base = (donnees.ndwi(s2) > alerte.BASELINE_T_NDWI) & donnees.pixels_utiles(s2, perms[k])
    res = (probs[meilleur][k] > 0.5) & donnees.pixels_utiles(s2, perms[k])
    fig, ax = plt.subplots(1, 4, figsize=(16, 4))
    ax[0].imshow(donnees.rgb(s2)); ax[0].set_title(f"{titre} : {noms[k]}")
    ax[1].imshow(base, cmap="Blues"); ax[1].set_title(f"Baseline : {parts['Baseline NDWI'][k]:.1%}")
    ax[2].imshow(res, cmap="Blues");  ax[2].set_title(f"{meilleur} : {parts[meilleur][k]:.1%}")
    ax[3].imshow(vrai, cmap="Blues"); ax[3].set_title(f"Annoté : {df['part_inondee'][k]:.1%}")
    for a in ax: a.axis("off")
    plt.show()

for nom in ["Ghana_313799", "Pakistan_1027214", "Pakistan_246718"]:
    if nom in noms:
        montrer(noms.index(nom), "Ratée par la baseline")

eval_ = df["partie"] != "train"
pred = parts[meilleur] >= seuils[meilleur]
erreurs = df[eval_ & (pred != df["alerte_vraie"])]
print(f"\nErreurs restantes de {meilleur} en validation et test : {len(erreurs)}")
for k in erreurs.index[:4]:
    montrer(k, "Fausse alerte" if pred[k] else "Manquée")

## À retenir

- Le réseau apprend à reconnaître l'eau à partir des **13 bandes**, là où le NDWI n'en utilise que deux.
- Le **choix entre les deux réseaux** se fait sur la validation (F1, fausses alertes) **et** sur le coût
  embarqué (taille, temps sur un cœur).
- Les poids sont enregistrés dans `modeles/` (session Colab) : prochaine étape, export ONNX et
  quantification pour réduire encore la taille.